# Masked Object Gaussian Splat Experiment

This notebook tests the new direction: instead of training one full-room Gaussian Splat and then assigning labels afterward, it creates a Nerfstudio dataset for one semantic object/class and trains an object-specific Gaussian Splat from masked image evidence.

This is an experiment notebook. It does **not** replace `pipeline.ipynb` yet.

Core idea:

```text
registered RGB frames + shared full-scene camera poses + target semantic masks
-> copied Nerfstudio dataset with every registered frame kept by default
-> per-frame mask_path, using empty masks when the target object is absent
-> optional object-only sparse point initialization
-> ns-train splatfacto
-> object_splat_experiments/<run_id>/exports/splat.ply
```

The masked training loss should only supervise pixels inside the target mask. The original full-room `splat.ply` is never overwritten.

Coordinate rule: object splats are trained in the same scene frame by keeping the original registered camera pose set by default. Do not compare multi-object placement from runs made with different pose subsets or different dataparser normalization settings.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 2. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime
import copy
import csv
import json
import logging
import math
import os
import re
import shutil
import subprocess
import sys

import cv2
import numpy as np
from tqdm.auto import tqdm

EXPERIMENT_VERSION = '2026-07-25-masked-object-splat-shared-frame-v2'

PROJECT_ROOT = Path('/content/drive/MyDrive/room-model-project')
SCAN_ID = 'scan_001'
SCAN_DIR = PROJECT_ROOT / 'scans' / SCAN_ID
SOURCE_NS_DATA_DIR = SCAN_DIR / 'nerfstudio-data'
DETECTION_DIR = SCAN_DIR / 'semantic_recon' / 'detections'
MASK_DIRS = [
    SCAN_DIR / 'semantic_recon' / 'segmentation' / 'masks',
    SCAN_DIR / 'semantic_recon' / 'layout' / 'masks',
]

# Start with one target label. Change this to 'desk', 'chair', 'floor', etc. after bed is proven.
TARGET_LABEL = 'bed'
TARGET_OBJECT_ID = None  # Optional: set to an object/detection id if your detection JSONs contain one.

# Frame and mask controls. Keep every registered frame by default so Nerfstudio computes the same
# scene normalization for every object run. Frames where the target object is absent receive an empty
# mask, so they preserve the camera pose distribution without training object appearance.
MAX_FRAMES = None  # Leave None for alignment with the full scene. Non-None is only for quick smoke tests.
KEEP_ALL_REGISTERED_FRAMES_FOR_SHARED_COORDINATES = True
EMPTY_MASK_FOR_MISSING_TARGET_FRAMES = True
MIN_MASK_AREA_FRACTION = 0.001
MASK_DILATION_PIXELS = 2
MASK_EROSION_PIXELS = 0
WRITE_MASKED_RGB_PREVIEWS = True
SHARED_COORDINATE_POLICY = 'reuse_full_scene_transforms_all_frames'

# Sparse initialization controls. This is important because a full-scene sparse point cloud can seed
# Gaussians outside the object. If object sparse init cannot be created, the notebook removes full-scene
# ply_file_path by default so training does not start from all-room COLMAP points.
CREATE_OBJECT_SPARSE_INIT = True
SPARSE_POINT_MIN_MASK_HITS = 1
SPARSE_POINT_CHUNK_SIZE = 200_000
SPARSE_POINT_MAX_OUTPUT_POINTS = 250_000
REMOVE_FULL_SCENE_PLY_INIT_IF_NO_OBJECT_INIT = True
CAMERA_CONVENTION = 'nerfstudio_opengl'  # Nerfstudio camera space: -Z forward, +Y up.

# Training controls. Increase TRAIN_ITERATIONS only after the dataset preview looks correct.
TRAIN_ITERATIONS = 3000
STEPS_PER_SAVE = 500
DOWNSCALE_FACTOR = 2
DOWNSCALE_FACTORS_TO_WRITE = sorted({1, int(DOWNSCALE_FACTOR)})

RUN_ID = f'{TARGET_LABEL}_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
EXPERIMENT_ROOT = SCAN_DIR / 'object_splat_experiments' / RUN_ID
OBJECT_NS_DATA_DIR = EXPERIMENT_ROOT / 'nerfstudio-data'
OBJECT_IMAGE_DIR = OBJECT_NS_DATA_DIR / 'images'
OBJECT_MASK_DIR = OBJECT_NS_DATA_DIR / 'masks'
OBJECT_PREVIEW_DIR = EXPERIMENT_ROOT / 'masked_rgb_preview'
OBJECT_TRAINING_DIR = EXPERIMENT_ROOT / 'training'
OBJECT_EXPORT_DIR = EXPERIMENT_ROOT / 'exports'
OBJECT_ALIGNMENT_MANIFEST = EXPERIMENT_ROOT / 'object_splat_alignment_manifest.json'
REPORT_DIR = SCAN_DIR / 'reports' / 'object_splat_experiment' / RUN_ID
FATAL_DIR = SCAN_DIR / 'reports' / 'fatal_errors'

for directory in [EXPERIMENT_ROOT, OBJECT_NS_DATA_DIR, OBJECT_IMAGE_DIR, OBJECT_MASK_DIR, OBJECT_PREVIEW_DIR, OBJECT_TRAINING_DIR, OBJECT_EXPORT_DIR, REPORT_DIR, FATAL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

LOG_PATH = REPORT_DIR / 'object_splat_experiment.log'
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s',
    handlers=[logging.FileHandler(LOG_PATH, mode='w'), logging.StreamHandler()],
)
logger = logging.getLogger('object_splat_experiment')

def fail(message):
    FATAL_DIR.mkdir(parents=True, exist_ok=True)
    fatal_path = FATAL_DIR / 'fatal_error.log'
    with fatal_path.open('a') as f:
        f.write(f'[{datetime.now().isoformat(timespec="seconds")}] object_splat_experiment: {message}\n')
    raise RuntimeError(message)

print('Experiment version:', EXPERIMENT_VERSION)
print('Target label:', TARGET_LABEL)
print('Run id:', RUN_ID)
print('Experiment root:', EXPERIMENT_ROOT)
logger.info('Experiment version: %s', EXPERIMENT_VERSION)
logger.info('Target label: %s; run id: %s', TARGET_LABEL, RUN_ID)

## 3. Install/Validate Dependencies

In [ ]:
import importlib.util

for package in ['plyfile']:
    if importlib.util.find_spec(package) is None:
        print(f'Installing {package}...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', package], check=True)

if shutil.which('ns-train') is None:
    print('Installing nerfstudio because ns-train was not found...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nerfstudio'], check=True)
else:
    print('nerfstudio already installed')

from plyfile import PlyData, PlyElement

print('ns-train:', shutil.which('ns-train'))
print('ns-export:', shutil.which('ns-export'))
if shutil.which('ns-train') is None:
    fail('ns-train is still unavailable after install attempt.')

## 4. Utility Functions

In [ ]:
def load_json(path):
    with Path(path).open('r') as f:
        return json.load(f)


def write_json(path, data):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(data, indent=2))



def safe_write_csv(path, rows, columns=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    records = list(rows)
    if columns is None:
        ordered = []
        for record in records:
            for key in record.keys():
                if key not in ordered:
                    ordered.append(key)
        columns = ordered
    with path.open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(columns or []))
        writer.writeheader()
        for record in records:
            writer.writerow({key: record.get(key, '') for key in columns})


def safe_read_csv(path):
    with Path(path).open('r', newline='') as f:
        return list(csv.DictReader(f))


def print_rows(rows, columns=None, limit=20):
    rows = list(rows)
    if columns is None:
        columns = []
        for row in rows[:limit]:
            for key in row.keys():
                if key not in columns:
                    columns.append(key)
    if not rows:
        print('(no rows)')
        return
    shown = rows[:limit]
    widths = {col: max(len(str(col)), *(len(str(row.get(col, ''))) for row in shown)) for col in columns}
    print(' | '.join(str(col).ljust(widths[col]) for col in columns))
    print('-+-'.join('-' * widths[col] for col in columns))
    for row in shown:
        print(' | '.join(str(row.get(col, '')).ljust(widths[col]) for col in columns))
    if len(rows) > limit:
        print(f'... {len(rows) - limit} more rows')


def describe_numeric(values):
    values = [float(value) for value in values if value is not None and str(value) != '' and np.isfinite(float(value))]
    if not values:
        return {'count': 0}
    arr = np.asarray(values, dtype=np.float64)
    return {
        'count': int(arr.size),
        'mean': float(arr.mean()),
        'std': float(arr.std()),
        'min': float(arr.min()),
        'p25': float(np.percentile(arr, 25)),
        'median': float(np.median(arr)),
        'p75': float(np.percentile(arr, 75)),
        'max': float(arr.max()),
    }


def normalize_label(label):
    return re.sub(r'[^a-z0-9]+', '_', str(label).strip().lower()).strip('_')


TARGET_LABEL_NORM = normalize_label(TARGET_LABEL)


def frame_key_from_any(value):
    stem = Path(str(value)).stem
    stem = re.sub(r'(_rgb|_image|_mask|_overlay)$', '', stem)
    return stem


def resolve_scan_path(value, base_dirs):
    if value is None:
        return None
    candidate = Path(str(value))
    if candidate.is_absolute() and candidate.exists():
        return candidate
    for base in base_dirs:
        path = Path(base) / candidate
        if path.exists():
            return path
    return candidate if candidate.is_absolute() else None


def frame_image_path(frame):
    file_path = frame.get('file_path')
    if not file_path:
        return None
    candidate = Path(file_path)
    if candidate.is_absolute() and candidate.exists():
        return candidate
    candidates = [
        SOURCE_NS_DATA_DIR / file_path,
        SOURCE_NS_DATA_DIR / 'images' / Path(file_path).name,
        SCAN_DIR / file_path,
    ]
    for path in candidates:
        if path.exists():
            return path
    return candidates[0]


def frame_intrinsics(frame, transforms_data, image_shape):
    h, w = image_shape[:2]
    fx = frame.get('fl_x', transforms_data.get('fl_x'))
    fy = frame.get('fl_y', transforms_data.get('fl_y', fx))
    cx = frame.get('cx', transforms_data.get('cx', w / 2.0))
    cy = frame.get('cy', transforms_data.get('cy', h / 2.0))
    if fx is None:
        camera_angle_x = frame.get('camera_angle_x', transforms_data.get('camera_angle_x'))
        if camera_angle_x is None:
            fail('Missing fl_x and camera_angle_x in transforms.json')
        fx = 0.5 * w / math.tan(0.5 * float(camera_angle_x))
    if fy is None:
        fy = fx
    return float(fx), float(fy), float(cx), float(cy)


def read_mask(mask_path, image_shape):
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise FileNotFoundError(f'Could not read mask: {mask_path}')
    h, w = image_shape[:2]
    if mask.shape[:2] != (h, w):
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
    mask = (mask > 0).astype(np.uint8)
    if MASK_DILATION_PIXELS > 0:
        kernel = np.ones((2 * int(MASK_DILATION_PIXELS) + 1, 2 * int(MASK_DILATION_PIXELS) + 1), dtype=np.uint8)
        mask = cv2.dilate(mask, kernel, iterations=1)
    if MASK_EROSION_PIXELS > 0:
        kernel = np.ones((2 * int(MASK_EROSION_PIXELS) + 1, 2 * int(MASK_EROSION_PIXELS) + 1), dtype=np.uint8)
        mask = cv2.erode(mask, kernel, iterations=1)
    return mask.astype(bool)



def suffixed_dataset_dir(base_dir, factor):
    base_dir = Path(base_dir)
    if int(factor) == 1:
        return base_dir
    return base_dir.parent / f'{base_dir.name}_{int(factor)}'


def write_dataset_image_mask_pyramid(image_path, image, mask, out_image_name, out_mask_name):
    h, w = image.shape[:2]
    written = []
    for factor in DOWNSCALE_FACTORS_TO_WRITE:
        factor = int(factor)
        image_dir = suffixed_dataset_dir(OBJECT_IMAGE_DIR, factor)
        mask_dir = suffixed_dataset_dir(OBJECT_MASK_DIR, factor)
        image_dir.mkdir(parents=True, exist_ok=True)
        mask_dir.mkdir(parents=True, exist_ok=True)
        out_image_path = image_dir / out_image_name
        out_mask_path = mask_dir / out_mask_name
        if factor == 1:
            shutil.copy2(image_path, out_image_path)
            mask_to_write = mask.astype(np.uint8) * 255
        else:
            target_w = max(1, int(round(w / factor)))
            target_h = max(1, int(round(h / factor)))
            resized_image = cv2.resize(image, (target_w, target_h), interpolation=cv2.INTER_AREA)
            resized_mask = cv2.resize(mask.astype(np.uint8), (target_w, target_h), interpolation=cv2.INTER_NEAREST)
            cv2.imwrite(str(out_image_path), resized_image)
            mask_to_write = resized_mask * 255
        cv2.imwrite(str(out_mask_path), mask_to_write)
        written.append({
            'factor': factor,
            'image_path': str(out_image_path),
            'mask_path': str(out_mask_path),
        })
    return written


def iter_detection_items(data):
    if isinstance(data, list):
        for item in data:
            if isinstance(item, dict):
                yield item
        return
    if not isinstance(data, dict):
        return
    for key in ['detections', 'objects', 'masks', 'segments', 'instances']:
        items = data.get(key)
        if isinstance(items, list):
            for item in items:
                if isinstance(item, dict):
                    yield item


def detection_label(item):
    for key in ['label', 'class', 'class_name', 'name', 'category']:
        if item.get(key):
            return normalize_label(item[key])
    return None


def detection_mask_path(item):
    for key in ['mask_path', 'mask_file', 'mask', 'segmentation_mask_path', 'path']:
        if item.get(key):
            return item[key]
    return None


def detection_object_id(item):
    for key in ['object_id', 'id', 'instance_id', 'track_id']:
        if item.get(key) is not None:
            return str(item[key])
    return None


def collect_target_masks_from_detections():
    frame_to_masks = {}
    rows = []
    json_paths = sorted(DETECTION_DIR.glob('*.json')) if DETECTION_DIR.exists() else []
    base_dirs = [PROJECT_ROOT, SCAN_DIR, SOURCE_NS_DATA_DIR, DETECTION_DIR, *MASK_DIRS]
    for json_path in json_paths:
        try:
            data = load_json(json_path)
        except Exception as exc:
            logger.warning('Skipping unreadable detection JSON %s: %s', json_path, exc)
            continue
        default_frame = data.get('frame') or data.get('image') or data.get('image_path') or json_path.stem
        frame_key = frame_key_from_any(default_frame)
        for item in iter_detection_items(data):
            label = detection_label(item)
            object_id = detection_object_id(item)
            if label != TARGET_LABEL_NORM:
                continue
            if TARGET_OBJECT_ID is not None and object_id != str(TARGET_OBJECT_ID):
                continue
            mask_path = resolve_scan_path(detection_mask_path(item), base_dirs)
            if mask_path is None or not mask_path.exists():
                logger.warning('Detection mask missing for %s in %s: %s', label, json_path, detection_mask_path(item))
                continue
            item_frame = item.get('frame') or item.get('image') or item.get('image_path') or default_frame
            item_frame_key = frame_key_from_any(item_frame)
            frame_to_masks.setdefault(item_frame_key, []).append(mask_path)
            rows.append({
                'frame_key': item_frame_key,
                'label': label,
                'object_id': object_id,
                'mask_path': str(mask_path),
                'source': str(json_path),
            })
    return frame_to_masks, rows


def collect_target_masks_by_filename(frame_keys):
    frame_to_masks = {}
    rows = []
    label_token = TARGET_LABEL_NORM
    for mask_dir in MASK_DIRS:
        if not mask_dir.exists():
            continue
        for mask_path in sorted(mask_dir.glob('*.png')):
            stem_norm = normalize_label(mask_path.stem)
            parts = stem_norm.split('_')
            if label_token not in parts and not stem_norm.endswith('_' + label_token) and f'_{label_token}_' not in f'_{stem_norm}_':
                continue
            matched_frame_key = None
            for frame_key in frame_keys:
                if mask_path.stem.startswith(frame_key):
                    matched_frame_key = frame_key
                    break
            if matched_frame_key is None:
                # Fallback for names like frame_000123_000_bed.png.
                match = re.match(r'(frame_\d+)', mask_path.stem)
                matched_frame_key = match.group(1) if match else mask_path.stem
            frame_to_masks.setdefault(matched_frame_key, []).append(mask_path)
            rows.append({
                'frame_key': matched_frame_key,
                'label': TARGET_LABEL_NORM,
                'object_id': None,
                'mask_path': str(mask_path),
                'source': 'filename_scan',
            })
    return frame_to_masks, rows


def merge_mask_maps(*maps):
    merged = {}
    seen = set()
    for mapping in maps:
        for frame_key, masks in mapping.items():
            for mask_path in masks:
                key = (frame_key, str(mask_path))
                if key in seen:
                    continue
                seen.add(key)
                merged.setdefault(frame_key, []).append(Path(mask_path))
    return merged

## 5. Validate Source Dataset And Find Target Masks

In [ ]:
TRANSFORMS_PATH = SOURCE_NS_DATA_DIR / 'transforms.json'
if not PROJECT_ROOT.exists():
    fail(f'Missing project root: {PROJECT_ROOT}')
if not SCAN_DIR.exists():
    fail(f'Missing scan dir: {SCAN_DIR}')
if not TRANSFORMS_PATH.exists():
    fail(f'Missing transforms.json: {TRANSFORMS_PATH}. Run reconstruction.ipynb first.')

transforms_data = load_json(TRANSFORMS_PATH)
frames = transforms_data.get('frames', [])
if not frames:
    fail(f'No frames in transforms.json: {TRANSFORMS_PATH}')

frame_keys = [frame_key_from_any(frame.get('file_path', f'frame_{idx:06d}')) for idx, frame in enumerate(frames)]
detection_mask_map, detection_rows = collect_target_masks_from_detections()
filename_mask_map, filename_rows = collect_target_masks_by_filename(frame_keys)
frame_to_masks = merge_mask_maps(detection_mask_map, filename_mask_map)

summary_rows = []
missing_images = 0
frames_with_masks = 0
for frame, frame_key in zip(frames, frame_keys):
    image_path = frame_image_path(frame)
    masks = frame_to_masks.get(frame_key, [])
    if image_path is None or not image_path.exists():
        missing_images += 1
    if masks:
        frames_with_masks += 1
    summary_rows.append({
        'frame_key': frame_key,
        'image_exists': bool(image_path is not None and image_path.exists()),
        'image_path': str(image_path),
        'mask_count': len(masks),
        'mask_paths': ';'.join(str(p) for p in masks),
    })

mask_source_rows = detection_rows + filename_rows
frame_mask_rows = summary_rows
safe_write_csv(REPORT_DIR / 'target_mask_sources.csv', mask_source_rows)
safe_write_csv(REPORT_DIR / 'target_frame_mask_summary.csv', frame_mask_rows)

print('Source frames:', len(frames))
print('Missing images:', missing_images)
print('Frames with target masks:', frames_with_masks)
print('Target mask records from detection JSON:', len(detection_rows))
print('Target mask records from filename scan:', len(filename_rows))
print('Reports:', REPORT_DIR)

if missing_images:
    fail(f'{missing_images} frame images referenced by transforms.json are missing.')
if frames_with_masks == 0:
    fail(f'No masks found for target label {TARGET_LABEL!r}. Check TARGET_LABEL, DETECTION_DIR, and MASK_DIRS.')

top_mask_rows = sorted(frame_mask_rows, key=lambda row: int(row.get('mask_count', 0)), reverse=True)[:20]
print_rows(top_mask_rows, columns=['frame_key', 'image_exists', 'mask_count', 'image_path'], limit=20)

## 6. Build Masked Nerfstudio Dataset

In [ ]:
new_transforms = copy.deepcopy(transforms_data)
new_frames = []
mask_stats = []
processed_count = 0
positive_mask_frame_count = 0
blank_mask_frame_count = 0
small_or_unreadable_mask_frame_count = 0

if MAX_FRAMES is not None and KEEP_ALL_REGISTERED_FRAMES_FOR_SHARED_COORDINATES:
    logger.warning(
        'MAX_FRAMES=%s is set. This is useful for smoke tests, but object splats from this run may not align with full-scene/object runs that use all frames.',
        MAX_FRAMES,
    )

for idx, (frame, frame_key) in enumerate(tqdm(list(zip(frames, frame_keys)), desc='Building object masked dataset')):
    if MAX_FRAMES is not None and processed_count >= MAX_FRAMES:
        break
    masks = frame_to_masks.get(frame_key, [])
    if not masks and not KEEP_ALL_REGISTERED_FRAMES_FOR_SHARED_COORDINATES:
        continue

    image_path = frame_image_path(frame)
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image is None:
        fail(f'Could not read frame image: {image_path}')
    image_shape = image.shape[:2]

    combined_mask = np.zeros(image_shape, dtype=bool)
    readable_masks = 0
    for mask_path in masks:
        try:
            combined_mask |= read_mask(mask_path, image_shape)
            readable_masks += 1
        except Exception as exc:
            logger.warning('Skipping unreadable mask %s: %s', mask_path, exc)

    area_fraction = float(combined_mask.mean())
    has_trainable_target_mask = readable_masks > 0 and area_fraction >= MIN_MASK_AREA_FRACTION

    if not has_trainable_target_mask:
        if not KEEP_ALL_REGISTERED_FRAMES_FOR_SHARED_COORDINATES or not EMPTY_MASK_FOR_MISSING_TARGET_FRAMES:
            mask_stats.append({
                'frame_key': frame_key,
                'status': 'skipped_too_small_or_unreadable',
                'readable_masks': readable_masks,
                'area_fraction': area_fraction,
                'source_masks': ';'.join(str(p) for p in masks),
            })
            small_or_unreadable_mask_frame_count += 1
            continue
        status = 'included_blank_shared_coordinate_frame' if not masks else 'included_blank_too_small_or_unreadable'
        if masks:
            small_or_unreadable_mask_frame_count += 1
        else:
            blank_mask_frame_count += 1
        combined_mask = np.zeros(image_shape, dtype=bool)
    else:
        status = 'included_positive_target_mask'
        positive_mask_frame_count += 1

    out_image_name = Path(frame.get('file_path', image_path.name)).name
    out_mask_name = f'{Path(out_image_name).stem}_{TARGET_LABEL_NORM}_mask.png'
    out_image_path = OBJECT_IMAGE_DIR / out_image_name
    out_mask_path = OBJECT_MASK_DIR / out_mask_name
    written_pyramid_files = write_dataset_image_mask_pyramid(
        image_path=image_path,
        image=image,
        mask=combined_mask,
        out_image_name=out_image_name,
        out_mask_name=out_mask_name,
    )

    if WRITE_MASKED_RGB_PREVIEWS:
        preview = image.copy()
        preview[~combined_mask] = 0
        cv2.imwrite(str(OBJECT_PREVIEW_DIR / out_image_name), preview)

    new_frame = copy.deepcopy(frame)
    new_frame['file_path'] = f'images/{out_image_name}'
    new_frame['mask_path'] = f'masks/{out_mask_name}'
    new_frame['source_frame_key'] = frame_key
    new_frame['target_label'] = TARGET_LABEL_NORM
    new_frame['has_target_mask'] = bool(has_trainable_target_mask)
    new_frame['shared_coordinate_frame'] = bool(KEEP_ALL_REGISTERED_FRAMES_FOR_SHARED_COORDINATES)
    new_frames.append(new_frame)
    processed_count += 1

    mask_stats.append({
        'frame_key': frame_key,
        'status': status,
        'readable_masks': readable_masks,
        'area_fraction': area_fraction,
        'has_target_mask': bool(has_trainable_target_mask),
        'source_masks': ';'.join(str(p) for p in masks),
        'output_image': str(out_image_path),
        'output_mask': str(out_mask_path),
        'written_downscale_factors': ','.join(str(item['factor']) for item in written_pyramid_files),
    })

if not new_frames:
    fail('No frames survived mask dataset creation. Lower MIN_MASK_AREA_FRACTION or check the mask paths.')
if positive_mask_frame_count == 0:
    fail(f'No positive target masks survived for {TARGET_LABEL!r}; refusing to train an all-empty object dataset.')

new_transforms['frames'] = new_frames
new_transforms['object_splat_experiment'] = {
    'version': EXPERIMENT_VERSION,
    'target_label': TARGET_LABEL_NORM,
    'target_object_id': TARGET_OBJECT_ID,
    'source_transforms': str(TRANSFORMS_PATH),
    'created_at': datetime.now().isoformat(timespec='seconds'),
    'shared_coordinate_policy': SHARED_COORDINATE_POLICY,
    'keep_all_registered_frames_for_shared_coordinates': KEEP_ALL_REGISTERED_FRAMES_FOR_SHARED_COORDINATES,
    'empty_mask_for_missing_target_frames': EMPTY_MASK_FOR_MISSING_TARGET_FRAMES,
    'original_frame_count': len(frames),
    'object_dataset_frame_count': len(new_frames),
    'positive_mask_frame_count': int(positive_mask_frame_count),
    'blank_mask_frame_count': int(blank_mask_frame_count),
    'small_or_unreadable_mask_frame_count': int(small_or_unreadable_mask_frame_count),
    'mask_dilation_pixels': MASK_DILATION_PIXELS,
    'mask_erosion_pixels': MASK_EROSION_PIXELS,
    'min_mask_area_fraction': MIN_MASK_AREA_FRACTION,
    'downscale_factor_for_training': int(DOWNSCALE_FACTOR),
    'downscale_factors_written': [int(value) for value in DOWNSCALE_FACTORS_TO_WRITE],
}

alignment_manifest = {
    'schema': 'object_splat_alignment_manifest.v1',
    'version': EXPERIMENT_VERSION,
    'created_at': datetime.now().isoformat(timespec='seconds'),
    'target_label': TARGET_LABEL_NORM,
    'run_id': RUN_ID,
    'coordinate_policy': SHARED_COORDINATE_POLICY,
    'source_transforms': str(TRANSFORMS_PATH),
    'object_transforms': str(OBJECT_NS_DATA_DIR / 'transforms.json'),
    'original_frame_count': len(frames),
    'object_dataset_frame_count': len(new_frames),
    'positive_mask_frame_count': int(positive_mask_frame_count),
    'blank_mask_frame_count': int(blank_mask_frame_count),
    'kept_all_registered_frames': bool(len(new_frames) == len(frames) and MAX_FRAMES is None),
    'downscale_factor_for_training': int(DOWNSCALE_FACTOR),
    'downscale_factors_written': [int(value) for value in DOWNSCALE_FACTORS_TO_WRITE],
    'expected_scene_transform_to_full_room_splat': {
        'translation': [0.0, 0.0, 0.0],
        'rotation_quaternion_xyzw': [0.0, 0.0, 0.0, 1.0],
        'scale': 1.0,
    },
    'warning': 'Object splats from runs with different frame subsets or dataparser pose normalization may not align. Use MAX_FRAMES=None and keep all registered frames for final object splats.',
}

mask_stats_rows = mask_stats
safe_write_csv(REPORT_DIR / 'masked_dataset_frame_report.csv', mask_stats_rows)
write_json(OBJECT_NS_DATA_DIR / 'transforms.json', new_transforms)
write_json(OBJECT_ALIGNMENT_MANIFEST, alignment_manifest)
write_json(REPORT_DIR / 'object_splat_alignment_manifest.json', alignment_manifest)

print('Object dataset frames:', len(new_frames), '/', len(frames))
print('Positive target-mask frames:', positive_mask_frame_count)
print('Blank shared-coordinate frames:', blank_mask_frame_count)
print('Small/unreadable target-mask frames:', small_or_unreadable_mask_frame_count)
print('Shared coordinate policy:', SHARED_COORDINATE_POLICY)
print('Alignment manifest:', OBJECT_ALIGNMENT_MANIFEST)
print('Object dataset:', OBJECT_NS_DATA_DIR)
print('Downscale folders written:', DOWNSCALE_FACTORS_TO_WRITE)
print('Training downscale factor:', DOWNSCALE_FACTOR)
print('Mask area summary for positive target frames:')
positive_area_values = [row['area_fraction'] for row in mask_stats_rows if row.get('status') == 'included_positive_target_mask']
print(json.dumps(describe_numeric(positive_area_values), indent=2))
print('Preview folder:', OBJECT_PREVIEW_DIR)

## 7. Optional: Create Object-Specific Sparse Point Initialization

In [ ]:
def resolve_ply_file_path_from_transforms(transforms_obj):
    value = transforms_obj.get('ply_file_path')
    candidates = []
    if value:
        p = Path(value)
        if p.is_absolute():
            candidates.append(p)
        else:
            candidates.extend([SOURCE_NS_DATA_DIR / p, SCAN_DIR / p])
    candidates.extend([
        SCAN_DIR / 'sparse_pc.ply',
        SOURCE_NS_DATA_DIR / 'sparse_pc.ply',
        SOURCE_NS_DATA_DIR / 'colmap' / 'sparse_pc.ply',
    ])
    for path in candidates:
        if path.exists():
            return path
    return None


def ply_xyz(vertex):
    names = set(vertex.dtype.names or [])
    if not {'x', 'y', 'z'}.issubset(names):
        fail('Sparse PLY is missing x/y/z fields.')
    return np.column_stack([
        np.asarray(vertex['x'], dtype=np.float32),
        np.asarray(vertex['y'], dtype=np.float32),
        np.asarray(vertex['z'], dtype=np.float32),
    ])


def project_points(points_chunk, frame, transforms_obj, image_shape):
    h, w = image_shape[:2]
    fx, fy, cx, cy = frame_intrinsics(frame, transforms_obj, image_shape)
    c2w = np.asarray(frame['transform_matrix'], dtype=np.float64)
    if c2w.shape != (4, 4):
        fail('Frame transform_matrix is not 4x4.')
    w2c = np.linalg.inv(c2w)
    homo = np.ones((len(points_chunk), 4), dtype=np.float64)
    homo[:, :3] = points_chunk.astype(np.float64)
    cam = (w2c @ homo.T).T[:, :3]
    if CAMERA_CONVENTION == 'nerfstudio_opengl':
        depth_z = -cam[:, 2]
        image_x = cam[:, 0]
        image_y = -cam[:, 1]
    elif CAMERA_CONVENTION == 'opencv':
        depth_z = cam[:, 2]
        image_x = cam[:, 0]
        image_y = cam[:, 1]
    else:
        fail(f'Unknown CAMERA_CONVENTION: {CAMERA_CONVENTION}')
    valid = np.isfinite(depth_z) & (depth_z > 0.05) & (depth_z < 50.0)
    u_float = np.full_like(depth_z, np.nan, dtype=np.float64)
    v_float = np.full_like(depth_z, np.nan, dtype=np.float64)
    u_float[valid] = fx * (image_x[valid] / depth_z[valid]) + cx
    v_float[valid] = fy * (image_y[valid] / depth_z[valid]) + cy
    valid &= np.isfinite(u_float) & np.isfinite(v_float)
    valid &= (u_float >= 0) & (u_float < w) & (v_float >= 0) & (v_float < h)
    idx = np.flatnonzero(valid)
    return idx, np.floor(u_float[idx]).astype(np.int32), np.floor(v_float[idx]).astype(np.int32)


def write_filtered_vertex_ply(source_ply, output_ply, keep_mask):
    ply = PlyData.read(str(source_ply))
    vertex = ply['vertex'].data
    filtered_vertex = vertex[keep_mask]
    elements = []
    for element in ply.elements:
        if element.name == 'vertex':
            elements.append(PlyElement.describe(filtered_vertex, 'vertex'))
        else:
            elements.append(element)
    PlyData(elements, text=False).write(str(output_ply))


object_sparse_written = False
object_sparse_point_count = 0
source_sparse_path = resolve_ply_file_path_from_transforms(transforms_data)

if not CREATE_OBJECT_SPARSE_INIT:
    print('Object sparse initialization disabled.')
elif source_sparse_path is None:
    print('No sparse point cloud found for object initialization.')
else:
    print('Source sparse PLY:', source_sparse_path)
    source_ply = PlyData.read(str(source_sparse_path))
    if 'vertex' not in source_ply:
        fail(f'No vertex element in sparse PLY: {source_sparse_path}')
    vertex = source_ply['vertex'].data
    sparse_points = ply_xyz(vertex)
    hit_counts = np.zeros(len(sparse_points), dtype=np.uint16)

    frame_lookup = {frame_key_from_any(f.get('file_path')): f for f in frames}
    included_report = safe_read_csv(REPORT_DIR / 'masked_dataset_frame_report.csv')
    included_rows = [row for row in included_report if str(row.get('status', '')).startswith('included')]
    for row in tqdm(included_rows, total=len(included_rows), desc='Filtering sparse points by target masks'):
        frame_key = row['frame_key']
        frame = frame_lookup.get(frame_key)
        if frame is None:
            continue
        image_path = frame_image_path(frame)
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image is None:
            continue
        mask_path = OBJECT_MASK_DIR / f'{Path(frame.get("file_path", image_path.name)).stem}_{TARGET_LABEL_NORM}_mask.png'
        if not mask_path.exists():
            continue
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE) > 0
        for start in range(0, len(sparse_points), SPARSE_POINT_CHUNK_SIZE):
            end = min(start + SPARSE_POINT_CHUNK_SIZE, len(sparse_points))
            local_idx, u, v = project_points(sparse_points[start:end], frame, transforms_data, image.shape[:2])
            if len(local_idx) == 0:
                continue
            hits = mask[v, u]
            if hits.any():
                hit_counts[start + local_idx[hits]] += 1

    keep = hit_counts >= SPARSE_POINT_MIN_MASK_HITS
    keep_count = int(keep.sum())
    if keep_count > SPARSE_POINT_MAX_OUTPUT_POINTS:
        rng = np.random.default_rng(42)
        keep_indices = np.flatnonzero(keep)
        selected = rng.choice(keep_indices, size=SPARSE_POINT_MAX_OUTPUT_POINTS, replace=False)
        keep = np.zeros(len(keep), dtype=bool)
        keep[selected] = True
        keep_count = int(keep.sum())

    object_sparse_path = OBJECT_NS_DATA_DIR / f'{TARGET_LABEL_NORM}_sparse_pc.ply'
    if keep_count > 0:
        write_filtered_vertex_ply(source_sparse_path, object_sparse_path, keep)
        object_sparse_written = True
        object_sparse_point_count = keep_count
        new_transforms['ply_file_path'] = object_sparse_path.name
        write_json(OBJECT_NS_DATA_DIR / 'transforms.json', new_transforms)
        print('Object sparse PLY written:', object_sparse_path)
        print('Object sparse points:', keep_count, '/', len(sparse_points))
    else:
        print('No sparse points projected into target masks.')

if not object_sparse_written and REMOVE_FULL_SCENE_PLY_INIT_IF_NO_OBJECT_INIT:
    new_transforms.pop('ply_file_path', None)
    write_json(OBJECT_NS_DATA_DIR / 'transforms.json', new_transforms)
    print('Removed full-scene ply_file_path from object transforms.json.')

sparse_summary = {
    'source_sparse_path': str(source_sparse_path) if source_sparse_path else None,
    'object_sparse_written': bool(object_sparse_written),
    'object_sparse_point_count': int(object_sparse_point_count),
    'remove_full_scene_ply_init_if_no_object_init': REMOVE_FULL_SCENE_PLY_INIT_IF_NO_OBJECT_INIT,
}
write_json(REPORT_DIR / 'object_sparse_init_summary.json', sparse_summary)
print(json.dumps(sparse_summary, indent=2))

## 8. Inspect The Dataset Before Training

In [ ]:
preview_files = sorted(OBJECT_PREVIEW_DIR.glob('*'))[:12]
print('Masked dataset transforms:', OBJECT_NS_DATA_DIR / 'transforms.json')
for factor in DOWNSCALE_FACTORS_TO_WRITE:
    image_dir = suffixed_dataset_dir(OBJECT_IMAGE_DIR, factor)
    mask_dir = suffixed_dataset_dir(OBJECT_MASK_DIR, factor)
    print(f'Images factor {factor}:', len(list(image_dir.glob('*'))), image_dir)
    print(f'Masks factor {factor}:', len(list(mask_dir.glob('*.png'))), mask_dir)
print('Preview images:', len(list(OBJECT_PREVIEW_DIR.glob('*'))))
print('First previews:')
for path in preview_files:
    print(path)

# Stop here and inspect preview images in Drive before running training.
# If the previews are not mostly the target object, fix segmentation/masks before training.
print('Alignment manifest:', OBJECT_ALIGNMENT_MANIFEST)
print('Coordinate policy:', SHARED_COORDINATE_POLICY)
print('For final aligned object splats, MAX_FRAMES should be None and object dataset frames should equal source frames.')

required_image_dir = suffixed_dataset_dir(OBJECT_IMAGE_DIR, DOWNSCALE_FACTOR)
required_mask_dir = suffixed_dataset_dir(OBJECT_MASK_DIR, DOWNSCALE_FACTOR)
if not required_image_dir.exists() or not any(required_image_dir.glob('*')):
    fail(f'Missing required training image folder/files for downscale factor {DOWNSCALE_FACTOR}: {required_image_dir}')
if not required_mask_dir.exists() or not any(required_mask_dir.glob('*.png')):
    fail(f'Missing required training mask folder/files for downscale factor {DOWNSCALE_FACTOR}: {required_mask_dir}')


## 9. Train Object-Specific Gaussian Splat

In [ ]:
LOCAL_OBJECT_DATA = Path(f'/content/object-splat-data-{RUN_ID}')
LOCAL_OBJECT_OUT = Path(f'/content/object-splat-training-{RUN_ID}')
LOCAL_LOG = Path(f'/content/ns_train_object_{RUN_ID}.log')

# Safety guard: if the sparse-init cell was skipped, do not silently train from a full-scene
# COLMAP point cloud. That would defeat the object-only experiment by seeding room-wide Gaussians.
if 'object_sparse_written' not in globals():
    object_sparse_written = False
    object_sparse_point_count = 0
    if REMOVE_FULL_SCENE_PLY_INIT_IF_NO_OBJECT_INIT:
        guarded_transforms = load_json(OBJECT_NS_DATA_DIR / 'transforms.json')
        guarded_transforms.pop('ply_file_path', None)
        write_json(OBJECT_NS_DATA_DIR / 'transforms.json', guarded_transforms)
        print('Sparse-init cell was skipped; removed full-scene ply_file_path before training.')

if LOCAL_OBJECT_DATA.exists():
    shutil.rmtree(LOCAL_OBJECT_DATA)
if LOCAL_OBJECT_OUT.exists():
    shutil.rmtree(LOCAL_OBJECT_OUT)
LOCAL_OBJECT_DATA.mkdir(parents=True, exist_ok=True)
LOCAL_OBJECT_OUT.mkdir(parents=True, exist_ok=True)

print('Copying masked object dataset to local Colab storage...')
subprocess.run(['rsync', '-a', '--info=progress2', f'{OBJECT_NS_DATA_DIR}/', f'{LOCAL_OBJECT_DATA}/'], check=True)

if shutil.which('nvidia-smi') is not None:
    subprocess.run(['nvidia-smi'], check=False)
else:
    print('WARNING: nvidia-smi was not found. This Colab runtime may not have a GPU attached.')
    print('In Colab, use Runtime > Change runtime type > Hardware accelerator > GPU before training.')

try:
    import torch
    print('torch.cuda.is_available():', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('CUDA device:', torch.cuda.get_device_name(0))
    else:
        print('WARNING: PyTorch does not see CUDA. ns-train splatfacto will likely fail or be unusably slow.')
except Exception as exc:
    print('WARNING: Could not import/check torch before training:', exc)

train_cmd = (
    f'MAX_JOBS=1 TORCH_CUDA_ARCH_LIST="7.5" ns-train splatfacto '
    f'--max-num-iterations {int(TRAIN_ITERATIONS)} '
    f'--steps-per-save {int(STEPS_PER_SAVE)} '
    f'--vis tensorboard '
    f'--data "{LOCAL_OBJECT_DATA}" '
    f'--output-dir "{LOCAL_OBJECT_OUT}" '
    f'nerfstudio-data --downscale-factor {int(DOWNSCALE_FACTOR)} '
    f'2>&1 | tee "{LOCAL_LOG}"'
)
print(train_cmd)
result = subprocess.run(['bash', '-lc', f'set -o pipefail; {train_cmd}'])
if result.returncode != 0:
    if LOCAL_LOG.exists():
        print('Training log tail:')
        print(''.join(LOCAL_LOG.read_text(errors='replace').splitlines(keepends=True)[-80:]))
    fail(f'Object splat training failed with exit code {result.returncode}. See {LOCAL_LOG}')

OBJECT_TRAINING_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run(['rsync', '-a', '--info=progress2', f'{LOCAL_OBJECT_OUT}/', f'{OBJECT_TRAINING_DIR}/'], check=True)
shutil.copy2(LOCAL_LOG, REPORT_DIR / LOCAL_LOG.name)

print('Training copied to:', OBJECT_TRAINING_DIR)
print('Training log:', REPORT_DIR / LOCAL_LOG.name)

## 10. Export Object Gaussian Splat

In [ ]:
config_paths = sorted(OBJECT_TRAINING_DIR.glob('**/config.yml'))
if not config_paths:
    fail(f'No config.yml found under {OBJECT_TRAINING_DIR}. Run the training cell first and confirm it copied outputs to Drive.')
CONFIG_PATH = config_paths[-1]
print('Config:', CONFIG_PATH)

checkpoint_paths = sorted(CONFIG_PATH.parent.glob('**/*.ckpt'))
if not checkpoint_paths:
    print('Files under config parent:')
    for path in sorted(CONFIG_PATH.parent.glob('**/*'))[:80]:
        print(path)
    fail(f'No Nerfstudio checkpoint .ckpt files found near {CONFIG_PATH}. Training likely did not finish/save a checkpoint.')
print('Latest checkpoint:', checkpoint_paths[-1])

LOCAL_OBJECT_EXPORT = Path(f'/content/object-splat-export-{RUN_ID}')
EXPORT_LOG = REPORT_DIR / f'ns_export_object_{RUN_ID}.log'

if LOCAL_OBJECT_EXPORT.exists():
    shutil.rmtree(LOCAL_OBJECT_EXPORT)
LOCAL_OBJECT_EXPORT.mkdir(parents=True, exist_ok=True)

export_cmd = (
    f'TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 ns-export gaussian-splat '
    f'--load-config "{CONFIG_PATH}" '
    f'--output-dir "{LOCAL_OBJECT_EXPORT}" '
    f'2>&1 | tee "{EXPORT_LOG}"'
)
print(export_cmd)
result = subprocess.run(['bash', '-lc', f'set -o pipefail; {export_cmd}'])
if result.returncode != 0:
    if EXPORT_LOG.exists():
        print('Export log tail:')
        print(''.join(EXPORT_LOG.read_text(errors='replace').splitlines(keepends=True)[-120:]))
    fail(f'Object splat export failed with exit code {result.returncode}. See {EXPORT_LOG}')

exported_local_files = sorted(str(p) for p in LOCAL_OBJECT_EXPORT.glob('**/*') if p.is_file())
if not exported_local_files:
    if EXPORT_LOG.exists():
        print('Export log tail:')
        print(''.join(EXPORT_LOG.read_text(errors='replace').splitlines(keepends=True)[-120:]))
    fail(f'ns-export finished but wrote no files to {LOCAL_OBJECT_EXPORT}. See {EXPORT_LOG}')

if OBJECT_EXPORT_DIR.exists():
    shutil.rmtree(OBJECT_EXPORT_DIR)
OBJECT_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run(['rsync', '-a', '--info=progress2', f'{LOCAL_OBJECT_EXPORT}/', f'{OBJECT_EXPORT_DIR}/'], check=True)

exported_files = sorted(str(p) for p in OBJECT_EXPORT_DIR.glob('**/*') if p.is_file())
if not exported_files:
    fail(f'Export files existed locally but did not sync to Drive: {OBJECT_EXPORT_DIR}')

print('Exported local files:')
for path in exported_local_files:
    print(path)
print('Exported Drive files:')
for path in exported_files:
    print(path)

## 11. Write Final Experiment Summary

In [ ]:
summary = {
    'schema': 'object_splat_experiment_summary.v1',
    'version': EXPERIMENT_VERSION,
    'created_at': datetime.now().isoformat(timespec='seconds'),
    'scan_id': SCAN_ID,
    'target_label': TARGET_LABEL_NORM,
    'target_object_id': TARGET_OBJECT_ID,
    'source_nerfstudio_data': str(SOURCE_NS_DATA_DIR),
    'object_nerfstudio_data': str(OBJECT_NS_DATA_DIR),
    'object_training_dir': str(OBJECT_TRAINING_DIR),
    'object_export_dir': str(OBJECT_EXPORT_DIR),
    'object_sparse_written': bool(object_sparse_written),
    'object_sparse_point_count': int(object_sparse_point_count),
    'shared_coordinate_policy': SHARED_COORDINATE_POLICY,
    'alignment_manifest': str(OBJECT_ALIGNMENT_MANIFEST),
    'keep_all_registered_frames_for_shared_coordinates': KEEP_ALL_REGISTERED_FRAMES_FOR_SHARED_COORDINATES,
    'empty_mask_for_missing_target_frames': EMPTY_MASK_FOR_MISSING_TARGET_FRAMES,
    'object_dataset_frame_count': int(len(new_frames)),
    'source_frame_count': int(len(frames)),
    'positive_mask_frame_count': int(positive_mask_frame_count),
    'blank_mask_frame_count': int(blank_mask_frame_count),
    'train_iterations': int(TRAIN_ITERATIONS),
    'downscale_factor': int(DOWNSCALE_FACTOR),
    'reports': str(REPORT_DIR),
}
write_json(REPORT_DIR / 'object_splat_experiment_summary.json', summary)
print(json.dumps(summary, indent=2))

print('\nMain file to download/check after export:')
print(OBJECT_EXPORT_DIR / 'splat.ply')
print('\nPreview masked RGB images before judging training:')
print(OBJECT_PREVIEW_DIR)